# TTS bootcamp – full pipeline (Colab, T4 GPU)

1. **Runtime → Change runtime type → T4 GPU.**
2. Add Colab secrets (key icon, left sidebar), with notebook access on:
   - `HF_TOKEN`: Hugging Face *read* token (access to ai4bharat/IndicF5 already granted).
   - `GH_TOKEN`: only if the GitHub repo is private (GitHub token with read access).
3. Set `ROLL` below, then **Runtime → Run all**.
4. If a cell says *restart*, do **Runtime → Restart session** and **Run all** again (happens once).
5. At the end, download `results_bundle.zip` (scores + logs, small) and the submission zip.

In [ ]:
ROLL = "ROLLNO"                       # your roll number
REPO = "/content/InterIITNLP2"
BRANCH = "claude/cool-tesla-oqlizq"
INDICF5_CODE = "13f7c4d627cc10111aea8fe9c0039462cacacdc7"   # github.com/ai4bharat/IndicF5 commit (code)
# The model weights stay pinned to the Hugging Face revision in kit/baseline/config.yaml

import os, subprocess, sys
def sh(cmd, check=True):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-4000:])
    if r.returncode:
        print(r.stderr[-6000:])
        if check:
            raise RuntimeError(cmd)
    return r

def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

if not os.path.isdir(f"{REPO}/submission"):
    tok = secret("GH_TOKEN")
    url = f"https://{tok}@github.com/Mihir269/InterIITNLP2.git" if tok else "https://github.com/Mihir269/InterIITNLP2.git"
    sh(f"git clone --depth 1 -b {BRANCH} {url} {REPO}")
else:
    sh(f"git -C {REPO} pull --ff-only", check=False)
os.chdir(f"{REPO}/submission")
print(os.getcwd(), os.listdir("."))

In [ ]:
# Install (keeps Colab's own torch). Skipped on the second run after a restart.
# IndicF5's setup.py asks for numpy<=1.26.4, but the kit needs numpy>=2.1 (and it works with it),
# so IndicF5 is installed with --no-deps and its dependencies are installed separately.
B = f"{REPO}/kit/baseline"
try:
    import f5_tts, speechbrain  # noqa: F401
    print("already installed")
except ImportError:
    C = f"-c {B}/constraints_colab.txt"
    sh(f"pip install -q -r {B}/requirements_colab.txt {C}")
    sh(f"pip install -q --no-deps git+https://github.com/ai4bharat/IndicF5.git@{INDICF5_CODE}")
    deps = ('accelerate cached_path click datasets "ema_pytorch>=0.5.2" "hydra-core>=1.3.0" jieba librosa '
            'matplotlib pydub pypinyin safetensors soundfile tomli torchdiffeq "tqdm>=4.65.0" '
            'transformers_stream_generator vocos wandb "x_transformers>=1.31.14" '
            '"audioop-lts; python_version>=\'3.13\'"')
    sh(f"pip install -q {deps} {C}")
    sh("pip check", check=False)   # a numpy<=1.26.4 complaint from f5-tts is expected and harmless
    print(">>> Installed. The runtime restarts itself now; when it reconnects, do Runtime > Run all again. <<<")
    os.kill(os.getpid(), 9)   # Colab restarts the kernel; the next Run all skips this install

In [ ]:
from huggingface_hub import login
login(token=secret("HF_TOKEN"))
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4 GPU!")

In [ ]:
# Load the model once; applies the kit's length fix (may ask for one restart)
sys.path.insert(0, f"{REPO}/kit")
from baseline import patch_length
print("length fix:", patch_length.ensure_patched() or "already patched")
from baseline.tts import get_engine
try:
    get_engine()
    print("IndicF5 loaded OK")
except RuntimeError as e:
    print(e)
    raise SystemExit(">>> Runtime > Restart session, then Run all again. <<<")

In [ ]:
# Quick smoke test: one item, listen to it
import csv
from IPython.display import Audio, display
from baseline.tts import synthesize, save_wav
p = {r["prompt_id"]: r for r in csv.DictReader(open(f"{REPO}/kit/data/prompts.csv", encoding="utf-8"))}["V1"]
y = synthesize("आज ऑफिस में इंटरनेट बहुत स्लो चल रहा है।", f"{REPO}/kit/data/" + p["file"], p["transcript"], seed=0)
save_wav("/content/demo.wav", y)
display(Audio("/content/demo.wav"))

In [ ]:
# Full pipeline: Task 1-4 audio, metrics, timing, results.json  (about 30-60 min on a T4)
# Free GPU time short? use e.g. SKIP="t3v t4" to skip the variant ablation and the speed bonus.
SKIP = ""
r = subprocess.run(f'KIT={REPO}/kit SKIP="{SKIP}" bash code/run_all.sh 2>&1 | tee /content/run_all.log',
                   shell=True, text=True)
print(open("/content/run_all.log").read()[-6000:])

In [ ]:
# Summary
import json, glob
for f in sorted(glob.glob("code/results/*.json")):
    if "candidates" in f: continue
    d = json.load(open(f))
    if "cer" in d:
        print(f"{os.path.basename(f):32} CER {d['cer']:.4f}  SIM {d['sim']:.4f}  failed {d['failed']}  by-cat {d['cer_by_category']}")

In [ ]:
# Package: submission zip + a small results bundle to send back for analysis
sh(f"python code/fill_results.py --roll {ROLL} --machine 'Colab T4'")
sh(f"python code/make_zip.py --roll {ROLL}")
sh(f"python {REPO}/kit/check_format.py {ROLL}_tts_bootcamp.zip", check=False)
sh(f"cd {REPO}/submission && zip -qr /content/results_bundle.zip code/results results.json /content/run_all.log code/*_timing.json", check=False)
from google.colab import files
files.download("/content/results_bundle.zip")
files.download(f"{REPO}/submission/{ROLL}_tts_bootcamp.zip")